In [1]:
%pip install langgraph langchain-core langchain-openai langchain-ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# ✅ Initialize LLM
llm = ChatOllama(model="gpt-oss:20b")  # or your preferred model

# ✅ Node function
def assistant(state: MessagesState):
    # Read conversation history
    messages = state["messages"]

    # Call LLM with history
    response = llm.invoke(messages)

    # Return new message (IMPORTANT: must be a list)
    return {
        "messages": [response]
    }

# ✅ Build graph
builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)

graph = builder.compile()

# ✅ Run graph
result = graph.invoke({
    "messages": [HumanMessage(content="Tell me a joke")]
})

# ✅ Get final output
print(result["messages"][-1].content)



Why don’t scientists trust atoms anymore?

Because they make up everything!


In [3]:

for m in result["messages"]:
    print(type(m).__name__, "=>", m.content)


HumanMessage => Tell me a joke
AIMessage => Why don’t scientists trust atoms anymore?

Because they make up everything!


In [14]:
state = {
    "messages": [HumanMessage(content="What is 2+2?")]
}

result1 = graph.invoke(state)
#print(result1["messages"][-1].content)
print(result1)

# Continue conversation
result2 = graph.invoke({
    "messages": result1["messages"] + [
        HumanMessage(content="Multiply that by 10")
    ]
})
print(result2)
#print(result2["messages"][-1].content)

for m in result2["messages"]:
    print({
        "type": type(m).__name__,
        "content": m.content,
        "tool_calls": getattr(m, "tool_calls", None)
    })


{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d264fa33-b535-44f8-967c-9d35e5b87bbe'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-05-31T12:51:30.4431372Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6077039700, 'load_duration': 231724400, 'prompt_eval_count': 74, 'prompt_eval_duration': 170491000, 'eval_count': 47, 'eval_duration': 5631723600, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019e7e16-ce4d-7460-a81d-381ecaee98be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 47, 'total_tokens': 121})]}
{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d264fa33-b535-44f8-967c-9d35e5b87bbe'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={}, response_metadata={'model': 'gpt-os

In [15]:
for m in result2["messages"]:
    print(type(m).__name__, "=>", m.content)


HumanMessage => What is 2+2?
AIMessage => 2 + 2 = 4.
HumanMessage => Multiply that by 10
AIMessage => \(4 \times 10 = 40\).
